# 11. Fine-Tuning Your First LLM (Hands-on)

Time to **actually fine-tune**! We put together everything: a small model + our dataset + LoRA,
and train with `SFTTrainer` (from the `trl` library).

> **Best run on Google Colab with a GPU.** On CPU it works for this tiny demo but is slow.

## The plan (5 simple steps)

| Step | What we do |
|------|-----------|
| 1 | Load a small **model** and its **tokenizer** |
| 2 | Load our **dataset** (the chat examples) |
| 3 | Set up **LoRA** |
| 4 | **Train** with `SFTTrainer` |
| 5 | **Save** the trained LoRA adapter |

## Step 1 - Load the model and tokenizer

In [ ]:
# !pip install transformers datasets peft trl accelerate
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model and tokenizer loaded.")

c:\Users\THIRU\OneDrive\Desktop\gen_ai_course\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 272/272 [00:00<00:00, 1582.23it/s]


Model and tokenizer loaded.


## Step 2 - A small dataset (chat format)

In [2]:
from datasets import Dataset

data = [
    {"messages": [
        {"role": "user", "content": "How long does delivery take?"},
        {"role": "assistant", "content": "Hi! GreenCart delivers within 2-3 business days."},
    ]},
    {"messages": [
        {"role": "user", "content": "Can I return an item?"},
        {"role": "assistant", "content": "Of course! You can return any item within 7 days for a full refund."},
    ]},
    {"messages": [
        {"role": "user", "content": "Do you deliver on Sundays?"},
        {"role": "assistant", "content": "We deliver Monday to Saturday. Sunday delivery is coming soon!"},
    ]},
]
dataset = Dataset.from_list(data)
print(dataset)

Dataset({
    features: ['messages'],
    num_rows: 3
})


## Step 3 - Set up LoRA

In [3]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

## Step 4 - Train with `SFTTrainer`

`SFTTrainer` handles the chat template, tokenization, and the training loop for us.
We keep it tiny (a few steps) so it finishes quickly.

In [4]:
import torch
from trl import SFTTrainer, SFTConfig

# Do we have a GPU? If not, we must tell the trainer to use the CPU.
has_gpu = torch.cuda.is_available()

training_args = SFTConfig(
    output_dir="finetuned_model",
    num_train_epochs=3,             # go over the data 3 times
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    logging_steps=1,                # print the loss every step
    report_to="none",               # no external logging
    use_cpu=not has_gpu,            # train on CPU when there is no GPU (e.g. a laptop)
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,        # <-- makes it LoRA fine-tuning
)

trainer.train()   # watch the loss go DOWN
print("Training done!")

Truncating train dataset: 100%|██████████| 3/3 [00:00<00:00, 236.97 examples/s]


Step,Training Loss
1,4.143539
2,4.095727
3,3.382455
4,3.357480
5,4.051775
6,4.003520
7,3.302693
8,3.935332
9,3.986665


Training done!


## Step 5 - Save the LoRA adapter

We only trained the small LoRA layers, so we save just those (the **adapter**). It is tiny.

In [5]:
trainer.save_model("finetuned_model")
print("Saved the LoRA adapter to the 'finetuned_model' folder.")

Saved the LoRA adapter to the 'finetuned_model' folder.


## Key points to remember

- Fine-tuning = **model + dataset + LoRA + trainer**.
- `SFTTrainer` handles the **chat template, tokenization, and training loop** for you.
- Pass `peft_config=lora_config` to make it **LoRA** (light) fine-tuning.
- Watch the **loss go down** - that means it is learning.
- You save a small **adapter**, not the whole model. Best run on a **GPU (Colab)**.